# 推理服务与量化补充线 · 第 1/8 课：Prefill、Decode 与 KV Cache 账本

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：能分别估算 prefill/decode 的计算形态和每请求 KV cache，并据此判断并发上限。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/` 已讲训练激活和并行度；本课关注推理请求生命周期、KV 常驻量与 TTFT/ITL，不重复 attention kernel。

前置：train 第 1～5 课、CUDA/Triton 基础、Transformer attention。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

自回归服务分为一次处理大量 prompt token 的 prefill，以及逐步生成少量 token 的 decode。KV cache 保存每层历史 K/V，避免每步重算全部前缀。

### 数据与控制如何流动

Prefill 把 prompt 批量送入模型并生成首个 KV；decode 每步读取已有 KV、追加一个位置。KV 大小近似 `2·L·B·T·H_kv·D·bytes`，其中 2 对应 K 与 V。

### 正确性条件与常见误区

必须使用 KV heads 而非 query heads；TP/CP 是否分片 KV 取决于具体部署。不能把模型权重空闲显存全部用于 KV，还要预留 workspace、碎片和运行时。

### 性能、成本与工程取舍

更长上下文和更高并发线性增大 KV；GQA/MQA、低精度 KV、分页和卸载可降容量，但可能增加精度损失或带宽压力。

## 具体演示

32 层、8 个 KV heads、head_dim=128、BF16、单请求 4096 token：KV 约 `2×32×4096×8×128×2=512 MiB`；并发 32 时仅 KV 就约 16 GiB。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐 KV cache 字节账本；显式拒绝负数维度。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def kv_cache_bytes(layers, batch, tokens, kv_heads, head_dim, bytes_per_value):
    values = (layers, batch, tokens, kv_heads, head_dim, bytes_per_value)
    if any(v < 0 for v in values):
        raise ValueError("dimensions must be non-negative")
    # TODO：K 和 V 各一份。
    return ______

assert kv_cache_bytes(32, 1, 4096, 8, 128, 2) == 536_870_912
assert kv_cache_bytes(32, 0, 4096, 8, 128, 2) == 0


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么 prefill 更像大 GEMM，而 decode 常成为 memory-bandwidth/小矩阵问题？

**你的答案：**


### Q2

一个 70B GQA 模型的 query heads 是 64、KV heads 是 8。若用 64 估算 KV，会造成什么决策错误？

**你的答案：**


### Q3

模型改用 TP=8 后，能否直接把单卡 KV 除以 8？还要确认什么？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
def kv_cache_bytes(layers, batch, tokens, kv_heads, head_dim, bytes_per_value):
    values = (layers, batch, tokens, kv_heads, head_dim, bytes_per_value)
    if any(v < 0 for v in values):
        raise ValueError("dimensions must be non-negative")
    return 2 * layers * batch * tokens * kv_heads * head_dim * bytes_per_value

assert kv_cache_bytes(32, 1, 4096, 8, 128, 2) == 536_870_912
assert kv_cache_bytes(32, 0, 4096, 8, 128, 2) == 0


### Q1 参考答案

Prefill 一次有较大的 token 维，可形成高并行、高算术强度 GEMM；decode 每请求每步通常只有一个新 token，矩阵 M 小，同时每步要读取权重和不断增长的 KV，因此更难喂满计算单元，带宽、批大小和调度更关键。

### Q2 参考答案

会把 KV 账本高估 8 倍，进而低估可服务并发、误判是否需要 KV 量化/卸载，并可能错误扩容。反过来若误用更小的值则会过度接纳并触发抢占或 OOM。

### Q3 参考答案

不能只按 TP 数盲除。要确认 attention/KV 的实际分片规则、GQA 的 KV head 数能否整除、是否有复制、上下文并行和框架的 KV layout；还要把 block metadata 与临时 buffer 计入实测峰值。

## 参考资料

- [PagedAttention / vLLM paper](https://arxiv.org/abs/2309.06180)
- [vLLM serving documentation](https://docs.vllm.ai/en/latest/cli/serve/)
- [TensorRT-LLM documentation](https://nvidia.github.io/TensorRT-LLM/)

API 与平台能力会演进；部署前应按目标版本重新核对。